<h1><b>Markov Decision Processes</b></h1>

<p align="justify">The problem you will examine is the Frozen Lake problem with a grid size of 8 x 8.</p>


<h2><b>Getting Familiar with the <i>Gymnasium</i> Library</b></h2>

In [ ]:
import numpy as np
np.bool8 = np.bool_
!pip install setuptools
!pip install gymnasium
import gymnasium as gym
from gym import wrappers

With the following command, you initialize the grid:



In [2]:
env = gym.make("FrozenLake-v1", map_name="8x8", render_mode="ansi")

With the following commands, you can reset the Agent to its initial position and visualize the grid and the Agent’s position.

In [ ]:
obs, info = env.reset()
print(env.render())

Next, you can define the next action randomly and the Agent takes one step.

In [ ]:
action = env.action_space.sample()
obs, reward, terminated, truncated, info = env.step(action)

print("Random action:", action)
print("New state:", obs)
print("Reward:", reward)
print("Terminated:", terminated)
print("Truncated:", truncated)
print(env.render())

<h2><b>Questions</b></h2>

<ul>
<li>After reading <a href="https://machinelearningjourney.com/index.php/2020/07/02/frozenlake/">this</a>, briefly describe the Frozen Lake problem as an optimization problem. What is the agent’s goal?</li>
<li>Briefly describe the <i>Policy Iteration</i> and <i>Value Iteration</i> algorithms, emphasizing the different way in which they approach solving the problem. Is it guaranteed that the two algorithms will converge to the optimal policy? If yes, do they lead to the same or to different optimal policies?</li>
<li>Run the program which solves the Frozen Lake problem using the <i>Policy Iteration</i> and <i>Value Iteration</i> algorithms, respectively. Which method converges to the optimal solution in fewer steps? Check the execution time of each program using the <i>time</i> command. What can you conclude regarding the complexity of each algorithm?</li>
</ul>

In [ ]:
import numpy as np
import gymnasium as gym


def run_episode(env, policy, gamma=1.0, max_steps=1000):
    obs, info = env.reset()
    total_reward = 0.0
    step_idx = 0

    while step_idx < max_steps:
        action = int(policy[obs])
        obs, reward, terminated, truncated, info = env.step(action)

        total_reward += (gamma ** step_idx) * reward
        step_idx += 1

        if terminated or truncated:
            break

    return total_reward, step_idx


def evaluate_policy(env, policy, gamma=1.0, n=5000):
    rewards = []
    steps = []

    for _ in range(n):
        r, s = run_episode(env, policy, gamma=gamma)
        rewards.append(r)
        steps.append(s)

    return np.mean(rewards), np.mean(steps)


def compute_policy_v(env, policy, gamma=0.99, eps=1e-10):
    base_env = env.unwrapped
    nS = env.observation_space.n
    v = np.zeros(nS, dtype=np.float64)

    while True:
        prev_v = v.copy()

        for s in range(nS):
            a = int(policy[s])
            v[s] = sum(
                p * (r + (0.0 if terminated else gamma * prev_v[s_]))
                for p, s_, r, terminated in base_env.P[s][a]
            )

        if np.max(np.abs(prev_v - v)) < eps:
            break

    return v


def extract_policy(env, v, gamma=0.99):
    base_env = env.unwrapped
    nS = env.observation_space.n
    nA = env.action_space.n
    policy = np.zeros(nS, dtype=int)

    for s in range(nS):
        q_sa = np.zeros(nA, dtype=np.float64)

        for a in range(nA):
            q_sa[a] = sum(
                p * (r + (0.0 if terminated else gamma * v[s_]))
                for p, s_, r, terminated in base_env.P[s][a]
            )

        policy[s] = int(np.argmax(q_sa))

    return policy


def policy_iteration(env, gamma=0.99, max_iterations=1000):
    nS = env.observation_space.n
    nA = env.action_space.n
    policy = np.random.choice(nA, size=nS)

    for i in range(max_iterations):
        old_policy = policy.copy()
        v = compute_policy_v(env, policy, gamma=gamma)
        policy = extract_policy(env, v, gamma=gamma)

        if np.array_equal(policy, old_policy):
            return policy, v, i + 1

    return policy, v, max_iterations


def value_iteration(env, gamma=0.99, eps=1e-10, max_iterations=100000):
    base_env = env.unwrapped
    nS = env.observation_space.n
    nA = env.action_space.n
    v = np.zeros(nS, dtype=np.float64)

    for i in range(max_iterations):
        prev_v = v.copy()

        for s in range(nS):
            q_sa = np.zeros(nA, dtype=np.float64)

            for a in range(nA):
                q_sa[a] = sum(
                    p * (r + (0.0 if terminated else gamma * prev_v[s_]))
                    for p, s_, r, terminated in base_env.P[s][a]
                )

            v[s] = np.max(q_sa)

        if np.max(np.abs(prev_v - v)) < eps:
            return v, i + 1

    return v, max_iterations


def print_policy(policy, shape=(8, 8)):
    arrows = {0: "←", 1: "↓", 2: "→", 3: "↑"}
    grid = np.array([arrows[int(a)] for a in policy]).reshape(shape)
    for row in grid:
        print(" ".join(row))


if __name__ == "__main__":
    gamma = 0.99

    # Slippery 8x8 FrozenLake
    env = gym.make("FrozenLake-v1", map_name="8x8", is_slippery=True)

    # Policy Iteration
    pi_policy, pi_v, pi_iters = policy_iteration(env, gamma=gamma)
    pi_reward, pi_steps = evaluate_policy(env, pi_policy, gamma=1.0, n=5000)

    # Value Iteration
    vi_v, vi_iters = value_iteration(env, gamma=gamma)
    vi_policy = extract_policy(env, vi_v, gamma=gamma)
    vi_reward, vi_steps = evaluate_policy(env, vi_policy, gamma=1.0, n=5000)

    print("=== Policy Iteration ===")
    print("Iterations to converge:", pi_iters)
    print("Average reward:", pi_reward)
    print("Average steps :", pi_steps)
    print_policy(pi_policy)

    print("\n=== Value Iteration ===")
    print("Iterations to converge:", vi_iters)
    print("Average reward:", vi_reward)
    print("Average steps :", vi_steps)
    print_policy(vi_policy)

    print("\n=== Comparison ===")
    print("Same policy?       ", np.array_equal(pi_policy, vi_policy))
    print("Max |V_pi - V_vi| =", np.max(np.abs(pi_v - vi_v)))